# Phase 1: Spurgeon Continued Pretraining — Step 7: Training (Notebook B)

This notebook executes PEFT QLoRA continued pretraining on `unsloth/Qwen2.5-3B` using the Hugging Face Dataset compiled in Notebook A. It handles 4-bit quantization, sequence packing (for fast compute), and cross-session checkpoint resumption logic.

## 1. Install Dependencies

In [1]:
# Install Unsloth and specific patched versions for Kaggle environment
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-841q0eyz/unsloth_868869f830fc48de88a820c9f14a835a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-841q0eyz/unsloth_868869f830fc48de88a820c9f14a835a
  Resolved https://github.com/unslothai/unsloth.git to commit cf97faed9ff73af5e46c18a21efdad24ded876dc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.9 MB/s eta 0:00:00
 

## 2. Model & PEFT Setup

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
LORA_RANK = 32

# Load base model in 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "unsloth/Qwen2.5-3B",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype        = None,
    load_in_4bit = True,
)

# Apply the PEFT LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha   = 64,
    lora_dropout = 0, # Critical: set to 0 to enable fast custom Triton kernels
    bias         = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 3. Configure Dataset and Training Arguments

In [3]:
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk
import os
import shutil

# Kaggle mounts input datasets as read-only. SFTTrainer's internal dataset.map()
# attempts to write temporary cached files in the dataset folder, causing an OSError.
# To prevent this, we copy the dataset from /kaggle/input/ to the writable /kaggle/working/ first.
src_dataset_path = "/kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-dataset/spurgeon_dataset"
local_dataset_path = "/kaggle/working/spurgeon_dataset"

if not os.path.exists(local_dataset_path):
    print(f"Copying dataset from {src_dataset_path} to writable path {local_dataset_path}...")
    shutil.copytree(src_dataset_path, local_dataset_path)
else:
    print(f"Dataset already exists at writable path: {local_dataset_path}")

# Load the dataset from the writable local directory
dataset = load_from_disk(local_dataset_path)

output_dir = "/kaggle/working/checkpoints"

# RESUMPTION CONFIGURATION
# To resume: mount the output dataset of your previous run (e.g. Run 1) as an Input Dataset
# RUN_NUMBER = 1 # Update to 2, 3, etc. for subsequent runs
RUN_NUMBER = 2

# If resuming from a previous run output mounted as an input, specify the path here:
# Example: "/kaggle/input/datasets/rafaelvieira1/spurgeon-training-run-1/checkpoints/checkpoint-1500"
#PREV_RUN_CHECKPOINT = None 
PREV_RUN_CHECKPOINT = "/kaggle/input/datasets/rafaelvieira1/spurgeon-training-run-1/checkpoints/checkpoint-216"

# Define total target epochs (must be equal to the current run number)
TOTAL_TARGET_EPOCHS = RUN_NUMBER

training_args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,   # Effective batch size = 16 (2 * 8)
    num_train_epochs            = TOTAL_TARGET_EPOCHS,
    warmup_steps                = 100,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    logging_steps               = 50,
    eval_strategy               = "steps",
    eval_steps                  = 500,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 3,     # Keep the last 3 checkpoints to manage disk space
    output_dir                  = output_dir,
    seed                        = 42,
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LENGTH,
    packing                     = True,  # Packs multiple short texts into a single context window
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset["train"],
    eval_dataset       = dataset["test"],
    args               = training_args,
)

Copying dataset from /kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-dataset/spurgeon_dataset to writable path /kaggle/working/spurgeon_dataset...


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/3451 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/3451 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/35 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=8):   0%|          | 0/35 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


## 4. Launch Training

In [4]:
# Fix potential PicklingError with SFTConfig in Unsloth/TRL on Kaggle
import sys
import trl
if hasattr(trainer, "args") and trainer.args.__class__.__name__ == "SFTConfig":
    import trl.trainer.sft_config
    trl.trainer.sft_config.SFTConfig = trainer.args.__class__
    sys.modules["trl.trainer.sft_config"].SFTConfig = trainer.args.__class__
    trl.SFTConfig = trainer.args.__class__

if PREV_RUN_CHECKPOINT:
    if os.path.exists(PREV_RUN_CHECKPOINT):
        print(f"Resuming training from checkpoint: {PREV_RUN_CHECKPOINT}")
        trainer.train(resume_from_checkpoint=PREV_RUN_CHECKPOINT)
    else:
        raise FileNotFoundError(f"Checkpoint not found at: {PREV_RUN_CHECKPOINT}. "
                                "Please check the path or ensure the previous run's dataset is mounted.")
else:
    print("Starting training from scratch...")
    trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Resuming training from checkpoint: /kaggle/input/datasets/rafaelvieira1/spurgeon-training-run-1/checkpoints/checkpoint-216


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,451 | Num Epochs = 2 | Total steps = 432
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
432,2.226443,2.304272


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/checkpoints/checkpoint-432/tokenizer_config.json.


## 5. Save PEFT Adapter Weights

In [5]:
output_path = f"/kaggle/working/spurgeon_lora_epoch{RUN_NUMBER}"
print(f"Saving PEFT LoRA adapter weights to {output_path}...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print("Weights saved successfully.")

Saving PEFT LoRA adapter weights to /kaggle/working/spurgeon_lora_epoch2...


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/spurgeon_lora_epoch2/tokenizer_config.json.


Weights saved successfully.
